# Supervised Learning - Brain Tumor Dataset

## General Approach

This notebook is the first part of the **DR CVRIE** project: supervised learning on a medical imaging dataset.

### Objective
Train a model capable of **classifying medical images** (brain MRI) by the presence or type of tumor. This is a supervised classification problem: each image is associated with a known label, and the model must learn to generalize this mapping.

### Why this dataset?
The **Brain Tumor MRI Dataset** (available on Kaggle) meets all project constraints:
- It contains only images and their labels (no additional tabular data to manage)
- It represents a **multi-class classification** problem (glioma, meningioma, no_tumor, pituitary)
- The medical topic is serious, well documented, and the classes are interpretable
- The data volume is large enough to train a robust model

### Libraries used

| Library | Role in this project |
|---|---|
| `os` | Navigate the dataset directory tree (folders = labels) |
| `numpy` | Image array manipulation, vectorized computations |
| `pandas` | Build a structured DataFrame to explore metadata |
| `PIL (Pillow)` | Load, validate, and transform images |
| `scikit-learn` | Stratified train/val/test split |

These libraries are all **allowed** by the assignment and are the industry standard for this type of Python pipeline.


## 1. Loading the dataset and building metadata


### Dataset structure

The dataset is organized as a directory tree where **each subfolder corresponds to a label**:

```
dataset/
|-- glioma/          <- malignant glioma tumor
|-- meningioma/      <- meningeal tumor (often benign)
|-- no_tumor/        <- healthy brain, no tumor
`-- pituitary/       <- pituitary gland tumor
```

### Label explanations

- **glioma**: tumor of the central nervous system, often aggressive. Represents the most severe class.
- **meningioma**: tumor of the meninges (brain covering), usually benign but can cause neurological symptoms.
- **no_tumor**: MRI from a healthy patient - a "negative" class, essential for the model to learn to separate pathological from normal.
- **pituitary**: pituitary gland tumor, often benign but hormonally impactful.

These four classes correspond to a **multi-class classification** problem (not binary), which justifies using an appropriate loss function (categorical cross-entropy).


In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance

# Path to the local dataset (structure: dataset/<label>/<image.jpg>)
DATASET_PATH = './dataset'

# Target size for all images after resizing.
# 224x224 is a standard from CNN architectures (VGG, ResNet, etc.).
# This ensures that all model inputs have exactly the same dimension.
IMG_SIZE = (224, 224)
CONTRAST_FACTOR = 1.2

# Build a DataFrame from the dataset directory tree.
# We iterate over each folder (= label) and then each image file inside.
records = []
for label in os.listdir(DATASET_PATH):
    folder = os.path.join(DATASET_PATH, label)
    # Ignore any stray files at the root (e.g., .DS_Store)
    if not os.path.isdir(folder):
        continue
    for file in os.listdir(folder):
        records.append({
            'id': file,                                  # file name
            'label': label,                              # class (= folder name)
            'path': os.path.join(folder, file)           # absolute path for loading
        })

# The DataFrame makes it easy to explore metadata
# without loading all images into memory right away
df = pd.DataFrame(records)
print(df.head(10))
print(f'Shape: {df.shape}')


## 2. Removing corrupted images


### Why this step?

A dataset downloaded from the web can contain files that are **partially downloaded, truncated, or have a corrupted EXIF header**. If such an image is loaded during training, it would raise an exception and interrupt the pipeline.

Pillow's `Image.verify()` method reads only the file header (without decoding the full image) to validate integrity - so it is a **fast and memory-light** check.

> Note: `verify()` can only be called once on an Image object (it consumes the object). That is why we reopen the file inside the function instead of passing a PIL object.


In [ ]:
def is_valid(path):
    """
    Check integrity of an image file.
    Returns True if the image can be opened and verified, False otherwise.

    We use a try/except block because verify() raises an exception
    for any invalid file (corrupted, truncated, wrong format, etc.)
    """
    try:
        Image.open(path).verify()   # read header only, very fast
        return True
    except:
        return False                # corrupted or unreadable file

# Filtering: keep only valid images
# reset_index(drop=True) reindexes the DataFrame from 0 to N-1 after filtering
df = df[df['path'].apply(is_valid)].reset_index(drop=True)

print(f'Valid images kept: {len(df)}')
print(df[['id', 'label']].head())


## 3. Class balance analysis


### Why check balance?

An **imbalanced** dataset (e.g., 90% `no_tumor`, 10% `glioma`) would bias training: the model would learn to predict the dominant class most of the time, achieving high overall accuracy **without really learning to distinguish minority classes**.

This is exactly the example cited in the assignment: *"If your data consists of 90% positive cases, a simple `return('Positive')` will be correct 90% of the time."*

If imbalance is detected, solutions include:
- **Oversampling** the minority class (duplication or data augmentation)
- **Undersampling** the majority class
- **Class weights** during training (penalize errors more on rare classes)


In [ ]:
import matplotlib.pyplot as plt

# Count number of images per class
class_counts = df['label'].value_counts()
print("Class distribution:")
print(class_counts)
print(f"
Max/min ratio: {class_counts.max() / class_counts.min():.2f}x")

# Bar plot visualization to spot imbalance
fig, ax = plt.subplots(figsize=(8, 4))
class_counts.plot(kind='bar', ax=ax, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'], edgecolor='black')
ax.set_title('Class Distribution - Brain Tumor Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Number of images')
ax.tick_params(axis='x', rotation=0)
# Display exact count above each bar
for i, v in enumerate(class_counts):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Image preprocessing


### Rationale for preprocessing choices

#### Grayscale conversion (`'L'`)

Brain MRIs are **grayscale medical images** by nature. The color channel carries no clinically relevant information - color in MRI comes only from visualization post-processing (false color). Converting to grayscale allows us to:
- **Reduce dimensionality**: an RGB image has 3 channels x 224 x 224 = ~150k values; in grayscale, it is 1 x 224 x 224 = ~50k. This divides the input vector size by 3.
- **Reduce noise**: colorization artifacts introduced by different MRI devices disappear.
- **Align with medical reality**: radiologists read these images in black and white.

#### Mild contrast enhancement

MRI scans coming from different sources can have slightly different intensity distributions. Applying a **small contrast boost** helps make tissue boundaries more visible while keeping the image medically plausible. A conservative factor such as `1.2` is used to:
- **Improve local structure visibility** in low-contrast scans
- **Reduce acquisition variability** between images from different devices or export settings
- **Preserve overall anatomy** by avoiding aggressive enhancement that could distort the data

#### Resize to 224x224 pixels

Images in the dataset have **heterogeneous resolutions**. A machine learning model requires **fixed-size** inputs. 224x224 is the standard resolution used by reference architectures (VGG16, ResNet50...), which would make potential transfer learning easier. It is a good tradeoff between retained detail and computational cost.

#### Pixel normalization to [0, 1]

Image pixels initially have integer values between **0 and 255**. Normalization divides by 255 to obtain floats in **[0, 1]**. This is essential for:
- **Numerical convergence** of algorithms (gradient descent works better with small values)
- Avoiding large-scale features dominating small-scale ones
- Compatibility with activation functions (sigmoid, ReLU) that operate on small values


In [ ]:
def preprocess(path):
    """
    Preprocessing pipeline for an image:
      1. Open the file
      2. Convert to grayscale ('L' = luminance in Pillow)
         -> removes the 3 RGB channels that are not useful for MRI
      3. Apply a mild contrast enhancement
         -> improves tissue boundary visibility without heavily altering the scan
      4. Resize to IMG_SIZE (224x224)
         -> makes all images the same size
      5. Convert to a float32 numpy array
         -> float32 (not float64) to save memory
      6. Normalize to [0, 1] by dividing by 255
         -> improves numerical stability during training
    """
    img = Image.open(path).convert('L')
    img = ImageEnhance.Contrast(img).enhance(CONTRAST_FACTOR)
    img = img.resize(IMG_SIZE)
    return np.array(img, dtype=np.float32) / 255.0

# Apply the pipeline to all dataset images
# X: array of shape (N, 224, 224) - N grayscale images
# y: array of corresponding labels (strings: 'glioma', 'meningioma', etc.)
X = np.array([preprocess(p) for p in df['path']])
y = df['label'].values

print(f'Shape of X: {X.shape}')     # expected: (N, 224, 224)
print(f'Unique labels: {np.unique(y)}')
print(f'Min pixel: {X.min():.3f} | Max pixel: {X.max():.3f}')  # check [0,1]


## 5. Train / Validation / Test split


### Split strategy and proportion rationale

We divide the dataset into **three distinct sets**:

| Set | Proportion | Role |
|---|---|---|
| **Train** | ~72% | Data on which the model fits its parameters |
| **Validation** | ~8% | Model evaluation at each epoch to detect overfitting and tune hyperparameters |
| **Test** | 20% | Final evaluation, **never seen during training or tuning** |

### Why 3 sets and not 2?

If we only had train/test, we would be tempted to tune hyperparameters (number of layers, learning rate...) based on the test score - which would **contaminate** the final evaluation. Validation acts as an intermediate test bench for decisions without touching the test set.

### Why `stratify=y`?

The `stratify` parameter ensures that **each class is represented proportionally** in the three sets. Without stratification, a random split could, by chance, under-represent a class in the test set, biasing evaluation. This is especially important if the dataset is slightly imbalanced.

### Why `random_state=42`?

Fixing the random seed ensures **reproducibility**. Two successive runs of the notebook will produce exactly the same splits.


In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: Split 80% (train+val) / 20% (test)
# stratify=y: each class is represented proportionally in each split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% of data for test
    stratify=y,          # preserves class distribution
    random_state=42      # reproducibility guaranteed
)

# Step 2: Split remaining train (80%) into 90% train / 10% val
# -> 10% of 80% = 8% of total -> final split: 72% / 8% / 20%
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,       # 10% of train -> ~8% of total dataset
    stratify=y_train,    # stratification applied here too
    random_state=42
)

# Check sizes
total = len(X_train) + len(X_val) + len(X_test)
print(f'Train : {len(X_train)} ({100*len(X_train)/total:.1f}%)')
print(f'Val   : {len(X_val)}   ({100*len(X_val)/total:.1f}%)')
print(f'Test  : {len(X_test)}  ({100*len(X_test)/total:.1f}%)')
print(f'Total : {total}')

# Verify stratification worked
print('
Distribution in train :', pd.Series(y_train).value_counts().to_dict())
print('Distribution in val   :', pd.Series(y_val).value_counts().to_dict())
print('Distribution in test  :', pd.Series(y_test).value_counts().to_dict())


## 6. Saving the cleaned dataset


### Why save as `.npz`?

The `.npz` format (NumPy compressed archive) is ideal here because:
- It **groups all arrays** (X_train, y_train, etc.) in a single file
- The `compressed` option significantly reduces disk size
- Reloading is **instant** in the next notebooks via `np.load()`
- It avoids **re-running the whole preprocessing pipeline** each session

This `.npz` file is the **starting point of the training notebook**.


In [ ]:
# Save the 6 arrays into a compressed .npz file
# The keys (X_train=, y_train=, etc.) will be the names used when reloading
np.savez_compressed(
    'brain_tumor_cleaned.npz',
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test
)

print('Saved: brain_tumor_cleaned.npz')
print('
To reload in another notebook:')
print("  data = np.load('brain_tumor_cleaned.npz', allow_pickle=True)")
print("  X_train = data['X_train']  # shape (N, 224, 224)")
print("  y_train = data['y_train']  # label strings")


## 7. Preprocessing conclusion

### Summary of operations performed

1. **Structured loading**: walk the directory tree to associate each image with its label
2. **Cleaning**: remove corrupted images to avoid training errors
3. **Distribution analysis**: check class balance and identify potential bias
4. **Normalization**: convert to grayscale, resize uniformly to 224x224, normalize to [0, 1]
5. **Stratified split**: split into train/val/test while preserving class proportions
6. **Persistence**: compressed save for reuse in the training notebook

### Next step

The next notebook will load `brain_tumor_cleaned.npz` and compare several supervised classification algorithms:
- A simple **baseline** model (e.g., SVM or Random Forest on flattened pixels)
- A more advanced model (e.g., a convolutional neural network with scikit-learn or PyTorch)

Evaluation will be done on the **test set only at the end of the experiment**, with reporting of accuracy, confusion matrix, and per-class classification report.
